In [25]:
import numpy as np
import pandas as pd
import scipy.linalg
import scipy.optimize
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# Basic utilities
# ============================================================

def relu(x):
    return np.maximum(x, 0.0)


def trace_inner(A, B):
    """
    Frobenius inner product Tr(A^T B).
    For symmetric matrices this is Tr(AB).
    """
    return np.sum(A * B)


def generate_teacher(cutoff, gamma, d):
    cutoff = int(cutoff)
    vals = np.arange(1, cutoff + 1, dtype=float) ** (-gamma)

    if cutoff < d:
        vals = np.concatenate([vals, np.zeros(d - cutoff)])

    vals = vals / np.linalg.norm(vals) * np.sqrt(d)
    return np.diag(vals)


def generate_goe_samples(d, nsamples, seed=0):
    """
    Generate fixed GOE samples once, so the fixed-point map is deterministic.
    """
    rng = np.random.default_rng(seed)
    samples = []

    for _ in range(nsamples):
        Z = rng.standard_normal((d, d))
        Z = (Z + Z.T) / np.sqrt(2.0 * d)
        samples.append(Z)

    return samples


# ============================================================
# Hat equations
# ============================================================

def q_hat_eq(q, m, sigma, alpha, Q0, lloss, noise):
    denom = sigma + lloss / 4.0
    return 2.0 * alpha * (Q0 - 2.0 * m + q + noise / 2.0) / denom**2


def m_hat_eq(q, m, sigma, alpha, Q0, lloss, noise):
    denom = sigma + lloss / 4.0
    return 2.0 * alpha / denom


def sigma_hat_eq(q, m, sigma, alpha, Q0, lloss, noise):
    denom = sigma + lloss / 4.0
    return 2.0 * alpha / denom


def hats_from_overlaps(x, alpha, Q0, lloss, noise):
    q, m, sigma = x

    qh = q_hat_eq(q, m, sigma, alpha, Q0, lloss, noise)
    mh = m_hat_eq(q, m, sigma, alpha, Q0, lloss, noise)
    sh = sigma_hat_eq(q, m, sigma, alpha, Q0, lloss, noise)

    return qh, mh, sh


# ============================================================
# Empirical sigma equation
# ============================================================

def sigma_empirical_Anew(evals, denoised_evals, sigma_hat, tol=1e-12):
    """
    Faithful vectorized version of Anew:

        sig = 2 sum_i 1_{nu_i > 0} / sigma_hat
        sig += sum_i sum_{j: nu_j != nu_i}
               (f(nu_i) - f(nu_j)) / (nu_i - nu_j)

        return sig / d^2

    Important:
        evals are the full true eigenvalues of M.
        denoised_evals are zero on discarded modes if rank < d.
    """
    evals = np.asarray(evals)
    denoised_evals = np.asarray(denoised_evals)

    d = evals.size

    diff = evals[:, None] - evals[None, :]
    fdiff = denoised_evals[:, None] - denoised_evals[None, :]

    mask = np.abs(diff) > tol

    pair_sum = np.sum(fdiff[mask] / diff[mask])
    diag_term = 2.0 * np.sum(evals > 0.0) / sigma_hat

    return (diag_term + pair_sum) / d**2


# ============================================================
# One Anew-style denoising step
# ============================================================

def denoise_matrix_from_goe_Anew(
    Z,
    Sstar,
    q_hat,
    m_hat,
    sigma_hat,
    lreg,
    rank,
):
    """
    Efficient faithful version of Anew.

    Anew computes

        M = sqrt(q_hat) Z + m_hat Sstar - 2 lreg I

        D, O = eigh(M)

        Denoised_D = relu(D) / sigma_hat

        if rank < d:
            Denoised_D[:d-rank] = 0

        Denoised_M = O diag(Denoised_D) O^T

    The expensive part is that Anew diagonalizes the full matrix.

    Here:
      - if rank == d, we do one full eigh;
      - if rank < d, we compute:
            full eigenvalues for sigma_empirical;
            only top-rank eigenvectors for q, m, TrS.
    """
    d = Sstar.shape[0]
    rank = int(rank)
    rank = max(1, min(rank, d))

    M = np.sqrt(max(q_hat, 0.0)) * Z + m_hat * Sstar
    M = M - 2.0 * lreg * np.eye(d)

    if rank == d:
        evals, evecs = scipy.linalg.eigh(
            M,
            check_finite=False,
            overwrite_a=False,
        )

        denoised_evals = relu(evals) / sigma_hat
        Denoised_M = (evecs * denoised_evals) @ evecs.T

        trace_denoised = np.sum(denoised_evals)

        return Denoised_M, evals, denoised_evals, trace_denoised

    else:
        # Full true eigenvalues, needed for sigma_empirical exactly as in Anew.
        evals = scipy.linalg.eigvalsh(
            M,
            check_finite=False,
            overwrite_a=False,
        )

        denoised_evals = relu(evals) / sigma_hat
        denoised_evals[:d - rank] = 0.0

        # Only top-rank eigenvectors are needed to build Denoised_M.
        lo = d - rank
        hi = d - 1

        evals_top, evecs_top = scipy.linalg.eigh(
            M,
            subset_by_index=[lo, hi],
            check_finite=False,
            overwrite_a=False,
        )

        denoised_top = relu(evals_top) / sigma_hat

        # This should match denoised_evals[d-rank:], up to numerical ordering.
        Denoised_M = (evecs_top * denoised_top) @ evecs_top.T

        trace_denoised = np.sum(denoised_top)

        return Denoised_M, evals, denoised_evals, trace_denoised


# ============================================================
# Anew fixed-point map, including loss and regterm
# ============================================================

def fixed_point_map_Anew(
    x,
    *,
    Sstar,
    goe_samples,
    alpha,
    Q0,
    noise,
    lloss,
    lreg,
    kappa_stud,
):
    """
    Maps x = (q, m, sigma) to T(x), faithfully to Anew.

    Also returns:
        loss = delta^2 / (4 epsilon^2) + lreg * TrSoverd
        regterm = lreg * TrSoverd

    where Anew used

        TrSoverd = mean trace(Denoised_M / q_hat) / d.
    """
    d = Sstar.shape[0]
    rank = int(round(kappa_stud * d))
    rank = max(1, min(rank, d))

    q_hat, m_hat, sigma_hat = hats_from_overlaps(
        x,
        alpha=alpha,
        Q0=Q0,
        lloss=lloss,
        noise=noise,
    )

    qs = []
    ms = []
    sigmas = []
    TrS = []

    for Z in goe_samples:
        Denoised_M, evals, denoised_evals, trace_denoised = denoise_matrix_from_goe_Anew(
            Z=Z,
            Sstar=Sstar,
            q_hat=q_hat,
            m_hat=m_hat,
            sigma_hat=sigma_hat,
            lreg=lreg,
            rank=rank,
        )

        qs.append(trace_inner(Denoised_M, Denoised_M) / d)
        ms.append(trace_inner(Sstar, Denoised_M) / d)
        sigmas.append(sigma_empirical_Anew(evals, denoised_evals, sigma_hat))

        TrS.append(trace_denoised )

    q_new = np.mean(qs)
    m_new = np.mean(ms)
    sigma_new = np.mean(sigmas)

    TrSmean= np.mean(TrS)
    delta = np.sqrt(q_hat) / m_hat
    epsilon = 2.0 / m_hat

    regterm = lreg * TrSmean / d
    loss = delta**2 / (4.0 * epsilon**2) + regterm

    return np.array([q_new, m_new, sigma_new]), loss, regterm, TrSmean


def fixed_point_residual_Anew(
    x,
    *,
    Sstar,
    goe_samples,
    alpha,
    Q0,
    noise,
    lloss,
    lreg,
    kappa_stud,
):
    Tx, _, _, _ = fixed_point_map_Anew(
        x,
        Sstar=Sstar,
        goe_samples=goe_samples,
        alpha=alpha,
        Q0=Q0,
        noise=noise,
        lloss=lloss,
        lreg=lreg,
        kappa_stud=kappa_stud,
    )

    return Tx - x


# ============================================================
# Root-only solver, Anew version
# ============================================================

def solve_by_root_only_Anew(
    x0,
    *,
    Sstar,
    goe_samples,
    alpha,
    Q0,
    noise,
    lloss=1.0,
    lreg=1.0,
    kappa_stud=1.0,
    root_tol=1e-7,
    good_residual_tol=1e-5,
    verbose=False,
):
    """
    Root-only solver for the Anew fixed-point map.

    Solves

        F(q,m,sigma) = T(q,m,sigma) - (q,m,sigma) = 0.

    Returns:
        x, mse, loss, regterm, nfev, success, info
    """
    x0 = np.array(x0, dtype=float)

    def F(x):
        return fixed_point_residual_Anew(
            x,
            Sstar=Sstar,
            goe_samples=goe_samples,
            alpha=alpha,
            Q0=Q0,
            noise=noise,
            lloss=lloss,
            lreg=lreg,
            kappa_stud=kappa_stud,
        )

    sol = scipy.optimize.root(
        F,
        x0,
        method="hybr",
        tol=root_tol,
    )

    x = np.array(sol.x, dtype=float)

    Fx = F(x)
    residual_norm = np.linalg.norm(Fx)
    relative_residual = residual_norm / (np.linalg.norm(x) + 1e-12)

    mse = Q0 + x[0] - 2.0 * x[1]

    _, loss, regterm, TrS = fixed_point_map_Anew(
        x,
        Sstar=Sstar,
        goe_samples=goe_samples,
        alpha=alpha,
        Q0=Q0,
        noise=noise,
        lloss=lloss,
        lreg=lreg,
        kappa_stud=kappa_stud,
    )

    success = bool(
        np.all(np.isfinite(x))
        and np.isfinite(mse)
        and np.isfinite(loss)
        and relative_residual < good_residual_tol
    )

    info = {
        "method": "root",
        "root_success": bool(sol.success),
        "success_good": success,
        "message": sol.message,
        "nfev": sol.nfev,
        "residual_norm": residual_norm,
        "relative_residual": relative_residual,
        "Fx": Fx,
        "TrShat": TrS,
    }

    if verbose:
        print(
            f"root | kappa_stud={kappa_stud:.6e} | "
            f"success={success} | mse={mse:.8e} | "
            f"loss={loss:.8e} | regterm={regterm:.8e}",
            flush=True,
        )

        if not success:
            print("  Root message:", sol.message, flush=True)
            print("  F(x):", Fx, flush=True)

    return x, mse, loss, regterm, sol.nfev, success, info


# ============================================================
# Kappa sweep, Anew version
# ============================================================

def run_kappa_sweep_Anew(
    *,
    gamma=0.75,
    noise=0.5,
    d=400,
    n=4e4,
    lreg_tilde=0.1,
    nsamples=10,
    seed=0,
    q_init=0.77487223,
    m_init=0.66581969,
    sigma_init=0.091110693,
    nkappa=64,
    verbose=True
):
    """
    Efficient Anew-style sweep.

    Returns:
        kappa_list, mse_list, loss_list, regterm_list, overlap_list, df
    """
    alpha = n / d**2

    Sstar = generate_teacher(cutoff=d, gamma=gamma, d=d)
    Q0 = 1.0

    goe_samples = generate_goe_samples(
        d=d,
        nsamples=nsamples,
        seed=seed,
    )

    kappa_list = np.logspace(np.log10(d**(0.2)), np.log10(1.0 / d), nkappa, endpoint=False)
    kappa_list = np.flip(np.unique(np.round(kappa_list * d)) / d)

    mse_list = np.zeros(len(kappa_list))
    loss_list = np.zeros(len(kappa_list))
    regterm_list = np.zeros(len(kappa_list))
    overlap_list = np.zeros((len(kappa_list), 6))

    info_list = []

    x = np.array([q_init, m_init, sigma_init], dtype=float)

    for i, kappa_stud in enumerate(kappa_list):
        x, mse, loss, regterm, nfev, success, info = solve_by_root_only_Anew(
            x,
            Sstar=Sstar,
            goe_samples=goe_samples,
            alpha=alpha,
            Q0=Q0,
            noise=noise,
            lloss=1.0,
            lreg=lreg_tilde,
            kappa_stud=kappa_stud,
            root_tol=1e-7,
            good_residual_tol=1e-5,
            verbose=verbose,
        )

        q_hat, m_hat, sigma_hat = hats_from_overlaps(
            x,
            alpha=alpha,
            Q0=Q0,
            lloss=1.0,
            noise=noise,
        )

        mse_list[i] = mse
        loss_list[i] = loss
        regterm_list[i] = regterm

        overlap_list[i] = np.array([
            x[0],
            x[1],
            x[2],
            q_hat,
            m_hat,
            sigma_hat,
        ])

        rank = max(1, int(round(kappa_stud * d)))

        info_list.append({
            "kappa_stud": kappa_stud,
            "rank": rank,
            "mse": mse,
            "loss": loss,
            "regterm": regterm,
            "TrShat": info["TrShat"],
            "nfev": nfev,
            "success": success,
            "root_success": info["root_success"],
            "success_good": info["success_good"],
            "method": info["method"],
            "message": info["message"],
            "residual_norm": info["residual_norm"],
            "relative_residual": info["relative_residual"],
            "Fx_q": info["Fx"][0],
            "Fx_m": info["Fx"][1],
            "Fx_sigma": info["Fx"][2],
            "q": x[0],
            "m": x[1],
            "sigma": x[2],
            "q_hat": q_hat,
            "m_hat": m_hat,
            "sigma_hat": sigma_hat,
        })

        if verbose:
            print(
                f"done | rank={rank:4d} | kappa_stud={kappa_stud:.6e} | "
                f"mse={mse:.8e} | loss={loss:.8e} | success={success}",
                flush=True,
            )

    df = pd.DataFrame(info_list)

    return kappa_list, mse_list, loss_list, regterm_list, overlap_list, df

In [ ]:
import os

d = 400


for noise in [0.5]:
    for gamma in [0.75]:
        for alpha in [1.2]:
            n=int(d**alpha)
            kappa_teacher=1.0
            l=alpha/2-1
            lreg_tilde = d**l 
            foldername=f"data_ERMlowrank_ltilde/logd(n)={alpha:.3f}/logd(ltilde)_{l:.2f}"
            if not os.path.exists(foldername):
                os.makedirs(foldername,exist_ok=True)
            filename=f"{foldername}/ERM_lowrank_n_{n}_d_{d}_ltilde_{lreg_tilde:.2e}_kappa_star_{kappa_teacher}_noise_{noise}_gamma_{gamma:.2f}.csv"
    
    
            kappa_list, mse_list, loss_list, regterm_list, overlap_list, df = run_kappa_sweep_Anew(
                gamma=gamma,
                noise=noise,
                d=d,
                n=n,
                lreg_tilde=lreg_tilde,
                nsamples=3,   # reduce for faster runs, increase for better accuracy
                seed=0,
                q_init=0.77487223,  #initial overlap guesses, does not need to be changed in most cases
                m_init=0.66581969,
                sigma_init=2.91110693,
                nkappa=64,
                verbose=True
            )

 
            if os.path.exists(filename):
                print(f"File {filename} already exists. Append mode enabled.")
                df.to_csv(
                    filename,
                    index=False,
                    mode='a',
                    header=False
                )
            else:
                print(f"File {filename} does not exist. Creating new file.")
                df.to_csv(
                    filename,
                    index=False,
                )

root | kappa_stud=3.315000e+00 | success=True | mse=9.58232097e-01 | loss=5.80060561e-03 | regterm=5.10540677e-03
done | rank=1326 | kappa_stud=3.315000e+00 | mse=9.58232097e-01 | loss=5.80060561e-03 | success=True
root | kappa_stud=2.940000e+00 | success=True | mse=9.58232097e-01 | loss=5.80060560e-03 | regterm=5.10540676e-03
done | rank=1176 | kappa_stud=2.940000e+00 | mse=9.58232097e-01 | loss=5.80060560e-03 | success=True
root | kappa_stud=2.607500e+00 | success=True | mse=9.58232097e-01 | loss=5.80060560e-03 | regterm=5.10540676e-03
done | rank=1043 | kappa_stud=2.607500e+00 | mse=9.58232097e-01 | loss=5.80060560e-03 | success=True
root | kappa_stud=2.312500e+00 | success=True | mse=9.58232097e-01 | loss=5.80060560e-03 | regterm=5.10540676e-03
done | rank= 925 | kappa_stud=2.312500e+00 | mse=9.58232097e-01 | loss=5.80060560e-03 | success=True
root | kappa_stud=2.052500e+00 | success=True | mse=9.58232097e-01 | loss=5.80060560e-03 | regterm=5.10540676e-03
done | rank= 821 | kappa_s

In [28]:
4e3*128 / 2e5

2.56